# M35 训练样本：当前重构版物理 ALB 的 10 点重算验证

本 Notebook **不运行 MLP**。它从 M35 的原始训练集中选择 10 个样本，使用当前 `ALB` 包的公开接口重新建立主动润滑轴承并求解物理油膜力，然后比较：

- 历史训练标签：训练 CSV 中保存的无量纲 `fx, fy`；
- 当前重算结果：`ALB.build_bearing()` 和 `Bearing.calculate()` 返回的量纲力除以力尺度后的结果。

目标是检查 ALB 重构后，物理仿真输出是否仍与 M35 建模时使用的标签一致。

## 数据和计算契约

M35 使用 S8b/S0011 数据源。每行包含 `freq, ex, ey, vx, vy, sx, sy`，其中：

- `freq`：轴颈转频，单位 Hz；
- `ex, ey`：相对于径向间隙的无量纲位移；
- `vx, vy`：相对于 `clearance × angular_speed` 的无量纲速度；
- `sx, sy`：直接输入轴承的归一化阀芯位移；
- `fx, fy`：以 `supply_pressure × length × radius / 2` 为尺度的无量纲油膜合力。

物理重算时使用当前论文配置，但将控制模式改为 `external_spool`、将阀模型改为 `static`，确保 `sx, sy` 直接进入物理 ALB，而不是被 PID 或动态伺服阀重新生成。

In [1]:
from pathlib import Path
import hashlib
import json
import math
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display


def find_alb_main_root() -> Path:
    # Search upward so the notebook works from the repository root or its own directory.
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "ALB").is_dir() and (candidate / "tests").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("无法定位 ALB_MAIN 仓库根目录。")


REPO_ROOT = find_alb_main_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import ALB
from ALB import build_bearing, load_bearing_config

MODEL_NAME = (
    "M0035_s8b_s0011_enormlt0p85_vnormlt0p4_edotvlt0p195_"
    "base12_mlp_minmax01_gelu_adamw_p1000_20260609"
)
MODEL_DIR = REPO_ROOT.parent / "SURROGATE_TRAIN" / "models" / MODEL_NAME
DATA_DIR = (
    REPO_ROOT.parent
    / "SURROGATE_TRAIN"
    / "data"
    / "albnn_master_dataset_20260521"
    / "filtered"
    / "s8b_s0011_valid56515_enormlt0p85_vnormlt0p4_edotvlt0p195_train45212_valid11303_20260609"
)
TRAIN_CSV = DATA_DIR / "train_s8b_s0011_enormlt0p85_vnormlt0p4_edotvlt0p195_base12_20260609.csv"
SOURCE_METADATA = (
    REPO_ROOT.parent
    / "SURROGATE_TRAIN"
    / "data"
    / "albnn_master_dataset_20260521"
    / "sources"
    / "s8b_s0011_alb_data2_exey0to0p9_200000_seed20260607"
    / "s8b_s0011_alb_data2_exey0to0p9_metadata.json"
)
PAPER_CONFIG = Path(r"F:/BaiduSyncdisk/博士论文/PAPER_WORK/task/PAPER/config/alb12.json5")

required_files = [
    MODEL_DIR / "m0035_training_contract.json",
    TRAIN_CSV,
    SOURCE_METADATA,
    PAPER_CONFIG,
]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError("缺少验证所需文件：\n" + "\n".join(missing))

git_head = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True
).strip()
config_sha256 = hashlib.sha256(PAPER_CONFIG.read_bytes()).hexdigest()

print(f"ALB 版本：{ALB.__version__}")
print(f"当前 Git 提交：{git_head}")
print(f"训练数据：{TRAIN_CSV}")
print(f"论文配置：{PAPER_CONFIG}")
print(f"论文配置 SHA-256：{config_sha256}")

ALB 版本：0.4.5
当前 Git 提交：75f2c9c42ad6285ab4caa75cf7b853cecdb24606
训练数据：G:\ALB_PROJECTS\SURROGATE_TRAIN\data\albnn_master_dataset_20260521\filtered\s8b_s0011_valid56515_enormlt0p85_vnormlt0p4_edotvlt0p195_train45212_valid11303_20260609\train_s8b_s0011_enormlt0p85_vnormlt0p4_edotvlt0p195_base12_20260609.csv
论文配置：F:\BaiduSyncdisk\博士论文\PAPER_WORK\task\PAPER\config\alb12.json5
论文配置 SHA-256：3df742e3e996d41fa7e8e7e6f418383d28c9d1129823a56540c276e386814129


## 选择 10 个 M35 训练样本

为了覆盖从低载荷到高载荷的计算区间，而不是只检查相邻或容易收敛的点，先按历史训练标签的 `force_norm` 从小到大排序，再在整个排序序列上等间隔选择 10 个位置。

该选择规则是确定性的，包含历史训练集的最小力点和最大力点。它是覆盖性检查，不是用来估计总体平均误差的随机抽样。

In [2]:
training = pd.read_csv(TRAIN_CSV)
assert len(training) == 45_212, f"M35 训练集行数异常：{len(training)}"
assert bool(training[["fx", "fy", "force_norm"]].notna().all().all())

sorted_training = training.sort_values("force_norm", kind="stable")
selection_positions = np.linspace(0, len(sorted_training) - 1, 10, dtype=int)
selected = sorted_training.iloc[selection_positions].copy()
selected.insert(0, "training_row", selected.index.astype(int))
selected.insert(0, "case_id", np.arange(1, 11, dtype=int))
selected = selected.reset_index(drop=True)

sample_columns = [
    "case_id", "training_row", "sample_id", "freq",
    "ex", "ey", "vx", "vy", "sx", "sy",
    "lambda_value", "force_norm", "fx", "fy",
]
display(selected[sample_columns].style.format(precision=9))

,case_id,training_row,sample_id,freq,ex,ey,vx,vy,sx,sy,lambda_value,force_norm,fx,fy
0,1,41763,67491,9.784734584,0.525161726,0.266606717,0.077940703,0.166419210,-0.261589873,0.143491201,0.064223912,0.000700355,0.000562789,0.000416853
1,2,13451,49815,59.978493910,0.220953450,0.655435064,-0.299969321,0.080320208,-0.040013013,-0.568130711,0.393679920,0.079511860,-0.060898452,0.051122543
2,3,43385,173120,137.100153636,0.119035665,0.387610519,-0.201493260,-0.073761772,-0.514880781,-0.479507524,0.899882174,0.131872338,-0.020725295,0.130233543
3,4,41229,129879,35.490286600,0.016575795,0.116964572,0.015266920,0.336750299,0.390349825,0.029792275,0.232947049,0.193173837,-0.099734528,-0.165436257
4,5,5097,19648,13.518664338,0.479324070,0.050291846,-0.212684732,0.155704488,-0.145497236,-0.452836717,0.088732250,0.268144529,-0.181375007,0.197495811
5,6,31219,192887,147.562283242,0.514821201,0.550961628,-0.135262869,-0.183662662,0.300405517,-0.306462947,0.968552294,0.365985583,-0.347360365,0.115265885
6,7,45156,86777,72.895428659,0.257159797,0.642830211,-0.143873518,0.322502482,-0.563434440,-0.219782816,0.478462606,0.500649671,-0.280199808,-0.414895361
7,8,35180,120832,105.486608858,0.719820526,0.125681387,0.064997845,-0.184819711,0.060915393,0.415706772,0.692380835,0.694989583,-0.552991718,0.420963989
8,9,43758,6107,159.509646913,0.609597447,0.525689147,-0.066773881,0.180534508,-0.377437238,-0.426852989,1.046971021,1.030680282,-1.020679360,-0.143232281
9,10,117,54914,175.607394818,0.452538627,0.718875474,-0.030655296,0.259704923,0.108888227,-0.182456419,1.152631562,4.206753109,-3.079456396,-2.865958831


## 建立当前重构版 ALB 配置并核对物理尺度

当前论文配置默认是 PID 加二阶伺服阀，不能直接接收训练样本中的 `sx, sy`。因此通过不可变配置的 `with_overrides()` 同时完成以下修改：

- `control.mode = external_spool`；
- `control.gains = None`，移除仅适用于 PID 的参数；
- 整个 `valve` 配置替换为 `{model: static}`。

随后由当前配置重新推导 `lambda_per_hz, beta_nondim, lr, cq0, cq1, cq2` 和力尺度，并与 S8b/S0011 数据源元数据比较。只有这些固定物理量一致，标签对比才有意义。

In [3]:
base_config = load_bearing_config(PAPER_CONFIG).with_overrides(
    {
        "control.mode": "external_spool",
        "control.gains": None,
        "valve": {"model": "static"},
    }
)

film = base_config.spec["film"]
restrictors = base_config.spec["restrictors"]
thermal = base_config.spec["thermal"]

miu = float(film["viscosity"])
clearance = float(film["clearance"])
radius = float(film["radius"])
length = float(film["length"])
supply_pressure = float(film["supply_pressure"])
density = float(film["density"])
length_ratio = length / (2.0 * radius)
reference_flow = supply_pressure * clearance**3 / (12.0 * miu * length_ratio)
force_scale = supply_pressure * length * radius / 2.0
lambda_per_hz = (
    1.5 * miu * (2.0 * math.pi) * length**2
    / (supply_pressure * clearance**2)
)
beta_nondim = (
    float(thermal["beta"])
    * float(thermal["heat_partition"])
    * supply_pressure
    / (density * float(thermal["cp_lub"]))
)
cq0 = (
    float(restrictors["discharge_coefficient"])
    * float(restrictors["valve_area"])
    * math.sqrt(2.0 / density)
    * math.sqrt(supply_pressure)
    / reference_flow
)
cq1 = (
    density
    / (5.0 * (math.pi * clearance * float(restrictors["orifice_diameter"])) ** 2)
    * reference_flow**2
    / supply_pressure
)
pipe_resistance = (
    128.0
    * miu
    * float(restrictors["orifice_length"])
    / (math.pi * float(restrictors["orifice_diameter"]) ** 4)
)
cq2 = pipe_resistance * reference_flow / supply_pressure

current_derived = {
    "lambda_per_hz": lambda_per_hz,
    "beta_nondim": beta_nondim,
    "lr": length_ratio,
    "cq0": cq0,
    "cq1": cq1,
    "cq2": cq2,
    "force_scale": force_scale,
}
source_metadata = json.loads(SOURCE_METADATA.read_text(encoding="utf-8"))
historical_derived = source_metadata["derived_inputs"]

contract_rows = []
for name, current_value in current_derived.items():
    historical_value = float(historical_derived[name])
    contract_rows.append(
        {
            "parameter": name,
            "historical": historical_value,
            "current": current_value,
            "difference": current_value - historical_value,
        }
    )
contract_comparison = pd.DataFrame(contract_rows)
display(contract_comparison.style.format(precision=14))

np.testing.assert_allclose(
    contract_comparison["current"],
    contract_comparison["historical"],
    rtol=1e-12,
    atol=1e-14,
)
np.testing.assert_allclose(
    selected["lambda_value"].to_numpy(dtype=float),
    selected["freq"].to_numpy(dtype=float) * lambda_per_hz,
    rtol=1e-12,
    atol=1e-14,
)

print(f"当前控制模式：{base_config.control_mode}")
print(f"当前阀模型：{base_config.spec['valve']['model']}")
print(f"力尺度：{force_scale:.6f} N")
print("固定物理尺度与 S8b/S0011 历史数据契约一致。")

,parameter,historical,current,difference
0,lambda_per_hz,0.00656368465125,0.00656368465125,0.00000000000000
1,beta_nondim,0.10837155963303,0.10837155963303,0.00000000000000
2,lr,0.75000000000000,0.75000000000000,0.00000000000000
3,cq0,1.34571222915236,1.34571222915236,0.00000000000000
4,cq1,0.20818792014191,0.20818792014191,0.00000000000000
5,cq2,0.00977847970357,0.00977847970357,0.00000000000000
6,force_scale,8400.00000000000000,8400.00000000000000,0.00000000000000


当前控制模式：external_spool
当前阀模型：static
力尺度：8400.000000 N
固定物理尺度与 S8b/S0011 历史数据契约一致。


## 使用当前 ALB 逐点重新计算

每个样本都单独创建配置和轴承，避免前一个样本的热状态、压力场或控制状态污染后一个样本。转换关系为：

- `displacement = clearance × [ex, ey]`；
- `velocity = clearance × 2π × freq × [vx, vy]`；
- `spool = [sx, sy]`；
- `force_nondim = force_dimensional / force_scale`。

本单元记录每点的收敛状态、量纲力、无量纲力、运行时间和警告数量。每个样本在新的 Python 子进程中建立并计算轴承，避免任何全局求解器资源或上一个轴承对象影响下一个样本。某一点失败时会保留错误信息并继续检查其余样本。

In [4]:
PHYSICAL_WORKER_CODE = r'''
import contextlib
import io
import json
import math
from pathlib import Path
import sys
import warnings

import numpy as np

payload = json.loads(sys.argv[1])
from ALB import build_bearing, load_bearing_config

buffer = io.StringIO()
with contextlib.redirect_stdout(buffer), warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    config = load_bearing_config(Path(payload["paper_config"])).with_overrides(
        {
            "control.mode": "external_spool",
            "control.gains": None,
            "valve": {"model": "static"},
            "film.rotation_frequency_hz": float(payload["freq"]),
        }
    )
    film = config.spec["film"]
    clearance = float(film["clearance"])
    angular_speed = 2.0 * math.pi * float(payload["freq"])
    displacement = clearance * np.asarray([payload["ex"], payload["ey"]], dtype=float)
    velocity = clearance * angular_speed * np.asarray(
        [payload["vx"], payload["vy"]], dtype=float
    )
    result = build_bearing(config).calculate(
        displacement=displacement,
        velocity=velocity,
        time=0.0,
        spool=(float(payload["sx"]), float(payload["sy"])),
    )

force_dim = np.asarray(result.force, dtype=float)
force_nondim = force_dim / float(payload["force_scale"])
print(
    json.dumps(
        {
            "current_fx": float(force_nondim[0]),
            "current_fy": float(force_nondim[1]),
            "current_fx_N": float(force_dim[0]),
            "current_fy_N": float(force_dim[1]),
            "converged": bool(result.convergence.converged),
            "warning_count": len(caught),
            "captured_stdout": buffer.getvalue(),
        },
        ensure_ascii=False,
    )
)
'''

recalculated_rows = []
total_started = time.perf_counter()
child_environment = dict(__import__("os").environ)
child_environment.update(
    {
        "PYTHONDONTWRITEBYTECODE": "1",
        "PYTHONUTF8": "1",
        "PYTHONIOENCODING": "utf-8",
        "OMP_NUM_THREADS": "1",
        "MKL_NUM_THREADS": "1",
        "OPENBLAS_NUM_THREADS": "1",
        "NUMEXPR_NUM_THREADS": "1",
    }
)

for row in selected.itertuples(index=False):
    case_started = time.perf_counter()
    record = {
        "case_id": int(row.case_id),
        "training_row": int(row.training_row),
        "sample_id": int(row.sample_id),
        "freq": float(row.freq),
        "label_fx": float(row.fx),
        "label_fy": float(row.fy),
    }
    payload = {
        "paper_config": str(PAPER_CONFIG),
        "force_scale": force_scale,
        "freq": float(row.freq),
        "ex": float(row.ex),
        "ey": float(row.ey),
        "vx": float(row.vx),
        "vy": float(row.vy),
        "sx": float(row.sx),
        "sy": float(row.sy),
    }
    try:
        completed = subprocess.run(
            [sys.executable, "-c", PHYSICAL_WORKER_CODE, json.dumps(payload)],
            cwd=REPO_ROOT,
            env=child_environment,
            capture_output=True,
            text=True,
            encoding="utf-8",
            timeout=180,
            check=False,
        )
        if completed.returncode != 0:
            raise RuntimeError(
                f"子进程返回码 {completed.returncode}: {completed.stderr.strip()}"
            )
        child_result = json.loads(completed.stdout.strip().splitlines()[-1])
        record.update(child_result)
        record["error"] = ""
    except Exception as exc:
        record.update(
            {
                "current_fx": np.nan,
                "current_fy": np.nan,
                "current_fx_N": np.nan,
                "current_fy_N": np.nan,
                "converged": False,
                "warning_count": 0,
                "captured_stdout": "",
                "error": f"{type(exc).__name__}: {exc}",
            }
        )

    record["elapsed_s"] = time.perf_counter() - case_started
    recalculated_rows.append(record)
    print(
        f"[{row.case_id:02d}/10] training_row={row.training_row}, "
        f"converged={record['converged']}, elapsed={record['elapsed_s']:.2f}s"
    )

recalculated = pd.DataFrame(recalculated_rows)
total_elapsed_s = time.perf_counter() - total_started
print(f"10 点物理重算总时间：{total_elapsed_s:.2f} s")

[01/10] training_row=41763, converged=True, elapsed=21.11s
[02/10] training_row=13451, converged=True, elapsed=25.30s
[03/10] training_row=43385, converged=True, elapsed=22.35s
[04/10] training_row=41229, converged=True, elapsed=18.88s
[05/10] training_row=5097, converged=True, elapsed=23.93s
[06/10] training_row=31219, converged=True, elapsed=27.40s
[07/10] training_row=45156, converged=True, elapsed=24.79s
[08/10] training_row=35180, converged=True, elapsed=24.12s
[09/10] training_row=43758, converged=True, elapsed=26.09s
[10/10] training_row=117, converged=True, elapsed=21.73s
10 点物理重算总时间：235.71 s


## 比较标准和结果

同时报告绝对误差、向量相对误差和量纲误差。由于历史标签和当前计算可能来自不同 CPU/线性代数环境，判定不要求二进制逐位相等。逐分量一致性标准设为：

`absolute_difference <= 1e-5 + 1e-3 × abs(historical_label)`

也就是无量纲绝对容差 `1e-5`、相对容差 `0.1%`。结果表会保留所有真实差异；即使有样本不通过，Notebook 也会完成执行并明确报告，而不是隐藏失败。

In [5]:
ABSOLUTE_TOLERANCE = 1e-5
RELATIVE_TOLERANCE = 1e-3

label_force = recalculated[["label_fx", "label_fy"]].to_numpy(dtype=float)
current_force = recalculated[["current_fx", "current_fy"]].to_numpy(dtype=float)
difference = current_force - label_force
absolute_error_norm = np.linalg.norm(difference, axis=1)
label_force_norm = np.linalg.norm(label_force, axis=1)
relative_error_norm = np.divide(
    absolute_error_norm,
    label_force_norm,
    out=np.full_like(absolute_error_norm, np.nan),
    where=label_force_norm > 0.0,
)
component_limits = ABSOLUTE_TOLERANCE + RELATIVE_TOLERANCE * np.abs(label_force)
component_consistent = np.abs(difference) <= component_limits
row_consistent = np.all(component_consistent, axis=1) & recalculated["converged"].to_numpy(bool)

recalculated["difference_fx"] = difference[:, 0]
recalculated["difference_fy"] = difference[:, 1]
recalculated["absolute_error_norm"] = absolute_error_norm
recalculated["relative_error_norm"] = relative_error_norm
recalculated["absolute_error_N"] = absolute_error_norm * force_scale
recalculated["consistent"] = row_consistent

result_columns = [
    "case_id", "training_row", "sample_id", "freq",
    "label_fx", "current_fx", "difference_fx",
    "label_fy", "current_fy", "difference_fy",
    "absolute_error_norm", "relative_error_norm",
    "absolute_error_N", "converged", "consistent", "elapsed_s",
]
display(recalculated[result_columns].style.format(precision=10))

finite_mask = np.isfinite(absolute_error_norm)
summary = pd.DataFrame(
    {
        "metric": [
            "selected_samples",
            "converged_samples",
            "consistent_samples",
            "mean_absolute_error_norm",
            "max_absolute_error_norm",
            "mean_relative_error_norm",
            "max_relative_error_norm",
            "max_absolute_error_N",
            "total_elapsed_s",
        ],
        "value": [
            len(recalculated),
            int(recalculated["converged"].sum()),
            int(recalculated["consistent"].sum()),
            float(np.mean(absolute_error_norm[finite_mask])),
            float(np.max(absolute_error_norm[finite_mask])),
            float(np.mean(relative_error_norm[finite_mask])),
            float(np.max(relative_error_norm[finite_mask])),
            float(np.max(recalculated.loc[finite_mask, "absolute_error_N"])),
            total_elapsed_s,
        ],
    }
)
display(summary.style.format({"value": "{:.12g}"}))

failed_rows = recalculated.loc[~recalculated["consistent"], "training_row"].tolist()
if not failed_rows:
    print("结论：10 个样本均在设定容差内，当前重构版 ALB 与 M35 历史训练标签一致。")
else:
    print("结论：当前重构版 ALB 未完全复现 M35 历史训练标签。")
    print(f"未通过的 training_row：{failed_rows}")
    print("这表示物理求解结果存在可测漂移，不应解释为 MLP 推理误差。")

,case_id,training_row,sample_id,freq,label_fx,current_fx,difference_fx,label_fy,current_fy,difference_fy,absolute_error_norm,relative_error_norm,absolute_error_N,converged,consistent,elapsed_s
0,1,41763,67491,9.7847345844,0.0005627886,0.0005625575,-0.0000002311,0.0004168528,0.0004181103,0.0000012575,0.0000012785,0.0018255148,0.0107394721,True,True,21.1075490000
1,2,13451,49815,59.9784939102,-0.0608984521,-0.0608963785,0.0000020737,0.0511225433,0.0511174406,-0.0000051027,0.0000055080,0.0000692727,0.0462672016,True,True,25.2992461000
2,3,43385,173120,137.1001536357,-0.0207252949,-0.0207276651,-0.0000023702,0.1302335431,0.1302402622,0.0000067190,0.0000071249,0.0000540284,0.0598487564,True,True,22.3473751000
3,4,41229,129879,35.4902866003,-0.0997345279,-0.0996702835,0.0000642444,-0.1654362569,-0.1653565640,0.0000796929,0.0001023636,0.0005299041,0.8598542308,True,True,18.8812880000
4,5,5097,19648,13.5186643382,-0.1813750074,-0.1814154666,-0.0000404592,0.1974958107,0.1974639378,-0.0000318730,0.0000515057,0.0001920818,0.4326476634,True,True,23.9290100000
5,6,31219,192887,147.5622832421,-0.3473603649,-0.3473570927,0.0000032721,0.1152658846,0.1152234795,-0.0000424051,0.0000425311,0.0001162098,0.3572614894,True,True,27.4043085000
6,7,45156,86777,72.8954286591,-0.2801998084,-0.2837357983,-0.0035359900,-0.4148953612,-0.4224961120,-0.0076007507,0.0083829969,0.0167442373,70.4171738245,True,False,24.7880918000
7,8,35180,120832,105.4866088577,-0.5529917184,-0.5554873942,-0.0024956758,0.4209639888,0.4220697289,0.0011057401,0.0027296627,0.0039276312,22.9291670309,True,False,24.1215487000
8,9,43758,6107,159.5096469133,-1.0206793604,-1.0231923390,-0.0025129786,-0.1432322812,-0.1451455324,-0.0019132512,0.0031584160,0.0030643994,26.5306942767,True,False,26.0926709000
9,10,117,54914,175.6073948184,-3.0794563960,-3.1429640478,-0.0635076518,-2.8659588311,-2.9738232965,-0.1078644654,0.1251717409,0.0297549530,1051.4426239453,True,False,21.7349561000


,metric,value
0,selected_samples,10
1,converged_samples,10
2,consistent_samples,6
3,mean_absolute_error_norm,0.013965312832
4,max_absolute_error_norm,0.125171740946
5,mean_relative_error_norm,0.00562782323376
6,max_relative_error_norm,0.0297549529806
7,max_absolute_error_N,1051.44262395
8,total_elapsed_s,235.7089114


结论：当前重构版 ALB 未完全复现 M35 历史训练标签。
未通过的 training_row：[45156, 35180, 43758, 117]
这表示物理求解结果存在可测漂移，不应解释为 MLP 推理误差。
